In [0]:
display(spark.sql("SELECT current_user()"))

In [0]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random

# Configuration
start_date = datetime.now() - timedelta(days=365)
total_hours = 8760
data = []

current_time = start_date

while current_time < datetime.now():
    # 1. Simulate "Offline" Gaps (1% chance of a gap)
    if random.random() < 0.02:
        gap_size = random.choice([6, 12, 24, 72]) # Gap in hours
        current_time += timedelta(hours=gap_size)
        continue # Skip generating data for this period

    # 2. Determine if this specific row is corrupted (30% chance)
    is_corrupt = random.random() < 0.30
    
    # Base Clean Values
    voltage = round(random.uniform(3.2, 4.2), 2)
    temp = round(random.uniform(25.0, 30.0), 1)
    cycle = random.randint(1, 500)
    
    if is_corrupt:
        corruption_type = random.choice(['alpha', 'negative', 'null', 'float_error'])
        
        if corruption_type == 'alpha':
            # Alphanumeric corruption
            voltage = random.choice(["4.2V", "Low", "Err", "3,8"]) 
        elif corruption_type == 'negative':
            # Impossible negatives
            temp = -40.0
            cycle = -10
        elif corruption_type == 'null':
            # Missing values
            temp = None
            voltage = np.nan
        elif corruption_type == 'float_error':
            # Cycle count should be INT, make it a weird float
            cycle = 125.667

    data.append([current_time, voltage, temp, cycle, "BATT_01"])
    current_time += timedelta(hours=1) # Move to next hour

# Create DataFrame
df = pd.DataFrame(data, columns=['timestamp', 'voltage', 'temperature', 'cycle_count', 'battery_id'])
df['voltage'] = pd.to_numeric(df['voltage'], errors='coerce')

# Define your path
bronze_path = "abfss://battery-data@batteryhealthdatalake.dfs.core.windows.net/bronze/old_battery_data"

# Save as Parquet
# 1. Convert your Pandas DataFrame to a Spark DataFrame
spark_df = spark.createDataFrame(df.astype(str)) 

# 2. Define your path
bronze_path = "abfss://battery-data@batteryhealthdatalake.dfs.core.windows.net/bronze/old_battery_data"

# 3. Save using Spark (this uses your Cluster's permissions automatically)
spark_df.write.mode("overwrite").format("parquet").save(bronze_path)

print(f"Success! Data stored at: {bronze_path}")